# Version 1 — Simple LangChain RAG
**Project:** AI Placement Preparation Assistant using RAG

This is the **first and simplest** version. Everything is kept in memory (RAM). There is no saved database yet — that comes in V2 (FAISS) and V3 (ChromaDB).

**Data used:** `technical_interview_.pdf`, `resume_guidelines.pdf`, `hr_questions.pdf`, `aptitude.pdf`

**Colab secret needed:** `Gemini_API`

### Where this notebook sits in the roadmap

| # | Roadmap stage | Covered in this notebook |
|---|---|---|
| 1 | Document Loading | ✅ `PyPDFLoader` |
| 2 | Text Splitting / Chunking | ✅ `RecursiveCharacterTextSplitter` |
| 3 | Embeddings | ✅ Gemini embeddings |
| 4 | Vector Database | ✅ In-memory store (simple, not saved) |
| 5 | Retrieval | ✅ `as_retriever` |
| 6 | LLM / Gemini | ✅ Gemini chat model |
| 7 | Answer + Source/Page Citation | ✅ File + page number |
| 8 | Streamlit App | Later (`app.py`) |

### Pipeline in one picture
```
4 PDFs → Loader → Pages (Documents) → Splitter → Chunks → Embeddings
      → In-memory vector store → Retriever → Prompt + Gemini → Answer + Source
```

## Step 1 — Setup
📍 *Before the roadmap starts.*

**What:** install the libraries. **Why:** Colab does not have them by default.

- `langchain-community` → document loaders
- `langchain-text-splitters` → chunking
- `langchain-google-genai` → Gemini (embeddings + chat)
- `pypdf` → the engine that reads PDF files

In [ ]:
%pip install -q langchain langchain-community langchain-text-splitters langchain-google-genai pypdf python-dotenv

## Step 2 — Connect Gemini API key
**Why:** Gemini is a Google service, so it needs your key to know the request is yours. We store the key in Colab *Secrets* so it never appears in the notebook.

In [ ]:
import os
from getpass import getpass

api_key = None

# 1) Google Colab: secret named Gemini_API  (left sidebar -> key icon)
try:
    from google.colab import userdata
    api_key = userdata.get("Gemini_API")
except Exception:
    pass

# 2) VS Code / local: a .env file in the project folder (GOOGLE_API_KEY=...) or an environment variable
if not api_key:
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")

# 3) Last option: type it (the text is hidden)
if not api_key:
    api_key = getpass("Enter your Gemini API key: ")

if not api_key:
    raise ValueError("No Gemini API key was provided.")

os.environ["GOOGLE_API_KEY"] = api_key
print("API key loaded:", bool(os.environ.get("GOOGLE_API_KEY")))


## Step 3 — Check that the PDFs are present
**Why:** a wrong path is the most common beginner error. We check first and print a clear message instead of a long red error.

In [ ]:
import os

from pathlib import Path

# Works in Google Colab (PDFs uploaded to /content) AND in VS Code (project folder with data/).
if Path("/content/hr_questions.pdf").exists():
    DATA_DIR = "/content"
    WORK_DIR = "/content"                      # where saved indexes go in Colab
else:
    _here = Path.cwd()
    _project = next((p for p in [_here, _here.parent] if (p / "data" / "hr_questions.pdf").exists()), _here)
    DATA_DIR = str(_project / "data")
    WORK_DIR = str(_project / "vectorstores")  # saved indexes go here (ignored by git)
os.makedirs(WORK_DIR, exist_ok=True)
print("PDF folder :", DATA_DIR)

PDF_FILES = [
    f"{DATA_DIR}/technical_interview_.pdf",
    f"{DATA_DIR}/resume_guidelines.pdf",
    f"{DATA_DIR}/hr_questions.pdf",
    f"{DATA_DIR}/aptitude.pdf",
]

missing = [f for f in PDF_FILES if not os.path.exists(f)]

if missing:
    print("❌ These files were not found:")
    for f in missing:
        print("   ", f)
    print("Upload them to Colab (left sidebar → Files) and run this cell again.")
else:
    print("✅ All 4 PDFs found")

## Step 4 — Document Loading
📍 **Roadmap stage 1: Document Loading**

**What:** `PyPDFLoader` opens a PDF and gives us a list of `Document` objects. **One page = one Document.**

**Why:** The LLM and the vector database cannot read a PDF file. They need plain text. The loader converts file → text + metadata.

```
Document
├── page_content  → the text of the page
└── metadata      → {'source': file path, 'page': page number (starts at 0)}
```

**Key lines**
- `PyPDFLoader(path)` → creates a loader for one file
- `loader.load()` → reads the file and returns the list of Documents
- `all_documents.extend(docs)` → joins pages of all 4 PDFs into one list

In [4]:
from langchain_community.document_loaders import PyPDFLoader

all_documents = []

for path in PDF_FILES:
    loader = PyPDFLoader(path)
    docs = loader.load()                      # one Document per page
    print(f"{os.path.basename(path):28s} -> {len(docs)} page(s)")
    all_documents.extend(docs)

print("\nTotal Documents (pages):", len(all_documents))

/tmp/ipykernel_1400/1712772264.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


technical_interview_.pdf     -> 5 page(s)
resume_guidelines.pdf        -> 3 page(s)
hr_questions.pdf             -> 3 page(s)
aptitude.pdf                 -> 4 page(s)

Total Documents (pages): 15


**Expect:** each PDF shows its page count, and a total.

**Observe:** the number of Documents = the number of pages, not the number of files.

In [5]:
doc = all_documents[0]

print("Type          :", type(doc))
print("Metadata      :", doc.metadata)
print("Characters    :", len(doc.page_content))
print("\nFirst 300 characters of page_content:\n")
print(doc.page_content[:300])

Type          : <class 'langchain_core.documents.base.Document'>
Metadata      : {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-08T11:57:06+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-10-08T11:57:06+00:00', 'subject': '(unspecified)', 'title': 'Technical Interview Preparation Notes: AI, Machine Learning and NLP', 'trapped': '/False', 'source': '/content/technical_interview_.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}
Characters    : 2755

First 300 characters of page_content:

Technical Interview Preparation Notes: AI,
 Machine Learning and NLP
Section 1: Artificial Intelligence and Machine Learning Basics
What is Artificial Intelligence?
Artificial Intelligence is the field of building systems that perform tasks that normally need human
intelligence, such as understandin


**Observe:** `metadata['page']` starts from **0**. So page index 0 = page 1 for a human. We fix this when we show citations.

**Mentor may ask:** *What is inside a Document?* → `page_content` (text) and `metadata` (details about where it came from).

## Step 5 — Text Splitting / Chunking
📍 **Roadmap stage 2: Text Splitting**

**What:** cut every page into smaller overlapping pieces called **chunks**.

**Why:**
1. A page has many topics. If we embed the whole page, the meaning gets mixed and search becomes blurry.
2. Small chunks = more exact retrieval.
3. The LLM has a limit on how much text it can take.

**Parameters**
- `chunk_size=700` → each chunk has at most 700 characters
- `chunk_overlap=100` → the last 100 characters of a chunk are repeated at the start of the next, so a sentence cut at the border is not lost

`RecursiveCharacterTextSplitter` tries to cut at paragraph → line → space → character, so chunks stay readable.

> Each chunk **keeps the metadata** of its page. This is how we can show file + page later.

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,      # max characters in one chunk
    chunk_overlap=100,   # characters repeated between neighbouring chunks
)

chunks = text_splitter.split_documents(all_documents)

print("Pages  :", len(all_documents))
print("Chunks :", len(chunks))
print("\n--- Chunk 1 ---")
print(chunks[0].page_content)
print("\nMetadata:", chunks[0].metadata)

Pages  : 15
Chunks : 53

--- Chunk 1 ---
Technical Interview Preparation Notes: AI,
 Machine Learning and NLP
Section 1: Artificial Intelligence and Machine Learning Basics
What is Artificial Intelligence?
Artificial Intelligence is the field of building systems that perform tasks that normally need human
intelligence, such as understanding language, recognising images, making decisions and learning
from experience. Machine learning and deep learning are subsets of AI.
What is the difference between AI, Machine Learning and Deep Learning?
AI is the broad goal of making machines intelligent. Machine Learning is a subset of AI where
systems learn patterns from data instead of being explicitly programmed. Deep Learning is a subset

Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-08T11:57:06+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-10-08T11:57:06+00:00', 'subject': '(unspecified)', 'title': 'Techni

**Observe:** chunks > pages. The chunk metadata still has `source` and `page`.

**Try:** change `chunk_size` to 300 and then 1200. See how the chunk count changes.

## Step 6 — Embeddings
📍 **Roadmap stage 3: Embeddings**

**What:** convert text into a list of numbers (a vector).

**Why:** computers cannot compare *meaning* of text, but they can compare numbers. Texts with similar meaning get similar vectors.

*Example:* "What is overfitting?" and "Model performs well on training data but fails on new data" will have vectors that are close, even though they share few words.

`embed_query(text)` turns one string into a vector.

In [7]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

# Try it on one sentence
vector = embeddings.embed_query("What is overfitting?")
print("Vector length  :", len(vector))
print("First 5 numbers:", vector[:5])

Vector length  : 3072
First 5 numbers: [-0.006956061, -0.00035724926, 0.0017957747, -0.052621245, 0.009199971]


**Observe:** the vector has the same length for every text (e.g. 3072 numbers). The numbers themselves have no meaning for us — only the *distance* between vectors matters.

## Step 7 — In-memory Vector Store
📍 **Roadmap stage 4: Vector Database** (simplest form)

**What:** `InMemoryVectorStore` embeds all chunks and keeps the vectors in RAM.

**Why simple?** No extra install, no files. But when the notebook restarts, everything is gone and we must embed again. FAISS (V2) and ChromaDB (V3) solve this.

`from_documents(chunks, embeddings)` = for every chunk → create vector → store vector + chunk text + metadata.

In [8]:
from langchain_core.vectorstores import InMemoryVectorStore

vectorstore = InMemoryVectorStore.from_documents(chunks, embeddings)
print("✅ Vector store ready with", len(chunks), "chunks")

✅ Vector store ready with 53 chunks


## Step 8 — Retrieval
📍 **Roadmap stage 5: Retrieval**

**What:** the question is converted to a vector, and the store returns the **k most similar chunks**.

A **retriever** is a small wrapper around the vector store with one job: `question → relevant chunks`. `k=4` means "give me 4 chunks".

In [9]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

question = "What is the STAR method?"
retrieved_docs = retriever.invoke(question)

for i, d in enumerate(retrieved_docs, start=1):
    print(f"\n--- Result {i} | {os.path.basename(d.metadata['source'])} | page {d.metadata['page'] + 1} ---")
    print(d.page_content[:300])


--- Result 1 | hr_questions.pdf | page 2 ---
Section 2: Behavioural and Situational Questions
What is the STAR method?
STAR is a structure for answering behavioural questions. Situation describes the context. Task
explains your responsibility. Action describes what you personally did. Result explains the outcome,
ideally with numbers or a clea

--- Result 2 | hr_questions.pdf | page 2 ---
others.
How should I answer: Tell me about a time you failed.
Choose a genuine example, such as a missed deadline or a project that did not work as planned.
Take responsibility without blaming others, explain what you learned, and show how you changed
your approach afterward.
How should I answer: Ho

--- Result 3 | hr_questions.pdf | page 1 ---
or consistency. Support each with a short real example that shows the strength in action. Avoid
generic claims without proof.
How should I answer: What is your weakness?
Pick a real but manageable weakness that is not critical for the job, and explain the step

**Observe:** the top result should come from `hr_questions.pdf`. Retrieval found it by *meaning*, not by exact keyword.

**Mentor may ask:** *Is retrieval the same as answering?* → No. Retrieval only finds text. The LLM writes the answer.

## Step 9 — Gemini LLM (with automatic model fallback)
📍 **Roadmap stage 6: LLM**

`temperature=0` makes the model stick to the context and avoids creative guessing — important for a document-based assistant.

**Why a list of models?** The Gemini *free tier* allows only a few requests **per model per day** (your error said `limit: 20, model: gemini-3.8-flash`). The quota is counted **separately for each model**, so we give the notebook a list. If model 1 returns a `429 RESOURCE_EXHAUSTED` error, LangChain's `.with_fallbacks()` automatically tries model 2.

`max_retries=1` stops the library from silently retrying for many seconds when the quota is already finished.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Free-tier quota is counted PER MODEL. If the first model is exhausted (429),
# the next one in the list is tried automatically.
MODEL_CANDIDATES = [
    "gemini-3.5-flash-lite",   # cost-efficient model (worked in the V3 run)
    "gemini-3.6-flash",         # backup model (separate free quota)
    # add more model names here only if they appear in Google AI Studio
]


def make_llm(model_name):
    return ChatGoogleGenerativeAI(
        model=model_name,
        temperature=0,    # 0 = stay factual, do not get creative
        max_retries=1,    # do not wait minutes retrying when the quota is gone
        timeout=60,
    )


llms = [make_llm(m) for m in MODEL_CANDIDATES]
llm = llms[0].with_fallbacks(llms[1:])     # model 1, then model 2, ...

print("Models, in the order they are tried:", MODEL_CANDIDATES)

## Step 10 — Prompt (anti-hallucination)
📍 **Roadmap stage 6: LLM (prompt part)**

**Why a strict prompt?** Your project rule: *if the answer is not in the documents, say so — do not invent.* The prompt enforces this.

`prompt | llm | StrOutputParser()` is a **chain**: the output of each piece goes into the next. The `|` symbol means "then".

In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

NOT_FOUND = "I could not find the answer in the uploaded documents."

prompt = ChatPromptTemplate.from_template("""You are an AI Placement Preparation Assistant.

Answer the question using ONLY the context below.
If the answer is not in the context, reply exactly:
"{not_found}"
Do not use outside knowledge.

Context:
{context}

Question: {question}

Answer:""")

# prompt -> llm -> plain text
answer_chain = prompt | llm | StrOutputParser()

In [12]:
def format_context(docs):
    """Join retrieved chunks into one text block for the prompt."""
    return "\n\n".join(d.page_content for d in docs)


def format_sources(docs):
    """Unique 'file - page' list. PyPDFLoader pages start at 0, humans start at 1."""
    seen, lines = set(), []
    for d in docs:
        name = os.path.basename(d.metadata.get("source", "unknown"))
        page = d.metadata.get("page", 0) + 1
        key = (name, page)
        if key not in seen:
            seen.add(key)
            lines.append(f"{name} - page {page}")
    return lines

## Step 10b — Handle quota errors (429) safely
**Problem:** when the free quota is finished, Gemini returns `429 RESOURCE_EXHAUSTED` and the notebook crashes with a 23-frame red traceback. Re-running cells also wastes the few requests you have.

**What this cell adds**
1. **Cache** → asking the same question again costs **0** requests.
2. **Wait and retry once** → only for *per-minute* limits (Gemini says how many seconds to wait). A *per-day* limit cannot be fixed by waiting, so we stop.
3. **Friendly message** → explains what happened and what to do, and still shows the **retrieved sources** (retrieval uses the embedding quota, which is separate from the answer-generation quota).

In [ ]:
import re
import time


class QuotaExhausted(Exception):
    """Raised when every Gemini model in MODEL_CANDIDATES has run out of quota."""

    def __init__(self, wait_seconds=None):
        super().__init__("Gemini quota exhausted")
        self.wait_seconds = wait_seconds


answer_cache = {}       # (question, k) -> (answer, docs)  => repeated questions cost 0 requests
gemini_requests = 0     # generation requests sent in this notebook session


def is_quota_error(err):
    text = str(err)
    return "429" in text or "RESOURCE_EXHAUSTED" in text


def retry_delay_seconds(err):
    """Read the suggested wait (in seconds) from the error text. None if not given in seconds."""
    text = str(err)
    m = (re.search(r"retryDelay['\"]?:\s*['\"]?(\d+(?:\.\d+)?)s", text)
         or re.search(r"retry in (\d+(?:\.\d+)?)s", text))
    return float(m.group(1)) if m else None


def is_busy_error(err):
    """503 UNAVAILABLE = the model is busy for a moment. Waiting a few seconds usually fixes it."""
    text = str(err)
    return "503" in text or "UNAVAILABLE" in text or "high demand" in text


def generate_answer(question, context, max_wait=65):
    """Call Gemini. Retry once for short per-minute limits; raise QuotaExhausted otherwise."""
    global gemini_requests
    payload = {"context": context, "question": question, "not_found": NOT_FOUND}

    for attempt in (1, 2):
        try:
            gemini_requests += 1
            return answer_chain.invoke(payload)
        except Exception as err:
            if is_busy_error(err) and attempt == 1:
                print("⏳ Gemini is busy (503). Retrying once in 3 seconds...")
                time.sleep(3)
                continue
            if not is_quota_error(err):
                raise                                   # a different problem: show it
            wait = retry_delay_seconds(err)
            if attempt == 1 and wait is not None and wait <= max_wait:
                print(f"⏳ Per-minute limit reached. Waiting {wait:.0f}s, then retrying once...")
                time.sleep(wait + 1)
                continue
            raise QuotaExhausted(wait) from err         # per-day limit: waiting will not help

## Step 11 — Full RAG function: Answer + Source/Page
📍 **Roadmap stages 5 + 6 + 7**

`ask()` does everything: retrieve → build context → call Gemini → print answer → print sources.

It now uses the cache and the safe `generate_answer()` from Step 10b, so a quota error prints a clear message instead of crashing.

In [14]:
def ask(question, k=4):
    cache_key = (question.strip().lower(), k)

    if cache_key in answer_cache:                              # same question asked before
        answer, docs = answer_cache[cache_key]
        print("♻️  Answer taken from cache - no Gemini request used.\n")
    else:
        k_retriever = vectorstore.as_retriever(search_kwargs={"k": k})
        docs = k_retriever.invoke(question)                    # 1. retrieve
        context = format_context(docs)                         # 2. build context
        try:
            answer = generate_answer(question, context)        # 3. generate
        except QuotaExhausted:
            print("Question:", question)
            print("\n⚠️  Could not generate an answer: the Gemini quota is finished for every")
            print("   model in MODEL_CANDIDATES (the free tier allows very few requests per day).")
            print("\n   What you can do:")
            print("   1. Add another model name to MODEL_CANDIDATES in Step 9, then re-run Step 9 and Step 10.")
            print("   2. Create a new API key in a NEW project (Google AI Studio) and update the Colab secret.")
            print("   3. Turn on billing for the project (much higher limits).")
            print("   4. Or wait for the daily reset.")
            print("\nRetrieval still worked. Most relevant chunks found:")
            for d in docs:
                name = os.path.basename(d.metadata.get("source", "unknown"))
                page = d.metadata.get("page", 0) + 1
                snippet = d.page_content[:150].replace("\n", " ")
                print(f" - {name} - page {page}: {snippet}...")
            return
        except Exception as err:                               # any other error: short message
            print(f"❌ {type(err).__name__}: {str(err)[:300]}")
            return
        answer_cache[cache_key] = (answer, docs)

    print("Question:", question)
    print("\nAnswer:\n", answer)
    if NOT_FOUND not in answer:                                # 4. cite sources
        print("\nSources:")
        for s in format_sources(docs):
            print(" -", s)
    print(f"\n(Gemini requests sent this session: {gemini_requests})")


ask("What is the STAR method?")

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Question: What is the STAR method?

Answer:
 STAR is a structure for answering behavioural questions. Situation describes the context. Task explains your responsibility. Action describes what you personally did. Result explains the outcome, ideally with numbers or a clear impact. It keeps answers clear and focused.

Sources:
 - hr_questions.pdf - page 2
 - hr_questions.pdf - page 1

(Gemini requests sent this session: 1)


### Test with questions from each of your 4 categories

In [15]:
ask("What is the formula for profit percentage?")          # aptitude
print("\n" + "=" * 70 + "\n")
ask("What should a fresher resume include?")               # resume
print("\n" + "=" * 70 + "\n")
ask("How does linear regression work?")                    # technical

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Question: What is the formula for profit percentage?

Answer:
 Profit percentage equals (profit divided by cost price) multiplied by 100.

Sources:
 - aptitude.pdf - page 1
 - aptitude.pdf - page 2

(Gemini requests sent this session: 2)




/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Question: What should a fresher resume include?

Answer:
 A fresher resume should include contact details, a short career objective or summary, education, technical skills, projects, internships or experience, certifications, and achievements. It should be kept to one page.

Sources:
 - resume_guidelines.pdf - page 1
 - resume_guidelines.pdf - page 3

(Gemini requests sent this session: 3)




/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Question: How does linear regression work?

Answer:
 Linear regression models the relationship between input features and a continuous target using a straight line, or a hyperplane for many features. It finds the weights that minimise the mean squared error between predictions and actual values.

Sources:
 - technical_interview_.pdf - page 2
 - technical_interview_.pdf - page 1

(Gemini requests sent this session: 4)


### Test an out-of-scope question (must say *not found*)

In [16]:
ask("Who won the FIFA World Cup in 2018?")

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Question: Who won the FIFA World Cup in 2018?

Answer:
 I could not find the answer in the uploaded documents.

(Gemini requests sent this session: 5)


**Observe:** the assistant should reply *"I could not find the answer in the uploaded documents."* and show no sources. This is the safety behaviour your problem statement asks for.

## Step 12 — Interactive chat
Type `exit` to stop.

In [17]:
while True:
    q = input("\nYou: ")
    if q.strip().lower() in ["exit", "quit"]:
        print("Goodbye!")
        break
    ask(q)


You: exit
Goodbye!


## 🎓 Viva corner

**Q. What is RAG?**

RAG = Retrieval-Augmented Generation. First we *retrieve* relevant text from our documents, then we give it to the LLM so it *generates* an answer based on that text.

**Q. Why not send the whole PDF to Gemini?**

Documents can be large and costly, the model has a limit, and too much text makes answers worse. Retrieval sends only the useful chunks.

**Q. Why do we use chunk overlap?**

So that a sentence or idea cut at a chunk border still appears whole in the next chunk.

**Q. What does an embedding represent?**

The meaning of a text as a list of numbers. Similar meaning → similar numbers.

**Q. What is the limitation of InMemoryVectorStore?**

It lives in RAM only. Restart the notebook and it is gone. That is why we move to FAISS (saved to disk) and ChromaDB (persistent database).

**Q. How do you stop hallucination?**

Strict prompt (answer only from context), temperature 0, and a fixed 'not found' reply. Sources are shown so the user can verify.

**Q. What is the `429 RESOURCE_EXHAUSTED` error and how did you handle it?**

The Gemini free tier allows only a limited number of requests per model per day. When it is used up, the API answers with HTTP 429. We handled it with (1) a list of models with automatic fallback, (2) a cache so repeated questions cost nothing, (3) a short wait-and-retry for per-minute limits, and (4) a clear message that still shows the retrieved sources. For a real deployment, billing should be enabled.
